# AutoGluon — Heart Disease Prediction
### An End-to-End AutoML Case Study

> **Dataset:** UCI Heart Disease (Cleveland)  
> **Task:** Binary classification — predict presence of heart disease  
> **Framework:** AutoGluon 1.1.1  

---

This notebook walks through a complete AutoGluon pipeline:
1. Load and explore the UCI Heart Disease dataset
2. Train a scikit-learn baseline (runnable without AutoGluon)
3. Present AutoGluon results and leaderboard (pre-run outputs included)
4. Visualize feature importance, confusion matrix, and model comparisons
5. Critically reflect on AutoGluon's strengths and limitations

> **Note:** AutoGluon cells include pre-run outputs. To re-run them, install AutoGluon with `pip install autogluon shap`. All other cells run with standard libraries only.

## 0. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

import sklearn
print(f'pandas  {pd.__version__}')
print(f'numpy   {np.__version__}')
print(f'sklearn {sklearn.__version__}')

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2563eb', '#16a34a', '#dc2626', '#9333ea', '#ea580c']
sns.set_palette(PALETTE)

pandas  2.0.3
numpy   1.24.4
sklearn 1.3.2


## 1. Data Loading & Exploration

The **UCI Heart Disease (Cleveland)** dataset contains 303 patient records. The binary target indicates whether heart disease was diagnosed.

| Feature | Description |
|---------|-------------|
| `age` | Age in years |
| `sex` | 1 = male, 0 = female |
| `cp` | Chest pain type (0–3) |
| `trestbps` | Resting blood pressure (mm Hg) |
| `chol` | Serum cholesterol (mg/dl) |
| `fbs` | Fasting blood sugar > 120 mg/dl |
| `restecg` | Resting ECG results |
| `thalach` | Maximum heart rate achieved |
| `exang` | Exercise-induced angina |
| `oldpeak` | ST depression induced by exercise |
| `slope` | Slope of peak exercise ST segment |
| `ca` | Number of major vessels (0–3) |
| `thal` | Thalassemia type |
| `target` | **1 = heart disease, 0 = no disease** |

In [2]:
url = 'https://raw.githubusercontent.com/sharmaroshan/Heart-UCI-Dataset/master/heart.csv'
df = pd.read_csv(url)

print(f'Dataset shape: {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nTarget distribution:')
vc = df['target'].value_counts()
print(f'  1 (Heart Disease): {vc[1]}  ({vc[1]/len(df)*100:.1f}%)')
print(f'  0 (No Disease):    {vc[0]}  ({vc[0]/len(df)*100:.1f}%)')
display(df.head(3))

Dataset shape: (303, 14)
Missing values: 0

Target distribution:
  1 (Heart Disease): 165  (54.5%)
  0 (No Disease):    138  (45.5%)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1


In [3]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

X_train = train_df.drop('target', axis=1)
y_train = train_df['target']
X_test  = test_df.drop('target', axis=1)
y_test  = test_df['target']

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {len(train_df)} rows  |  Test: {len(test_df)} rows')

Train: 242 rows  |  Test: 61 rows


### EDA — Feature Distributions by Target Class

In [4]:
features_to_plot = ['age', 'thalach', 'chol', 'trestbps', 'oldpeak', 'ca', 'cp', 'thal']
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for ax, feat in zip(axes.flatten(), features_to_plot):
    for label, color, name in [(0, PALETTE[2], 'No Disease'), (1, PALETTE[0], 'Heart Disease')]:
        subset = df[df['target'] == label][feat]
        ax.hist(subset, alpha=0.6, bins=15, color=color, label=name, density=True)
    ax.set_title(feat, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Target Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('assets/eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Notable: cp, thal, ca, and thalach show strong class separation')

## 2. Scikit-learn Baseline

Before AutoGluon, we establish a tuned Random Forest baseline — the kind of pipeline a practitioner would build manually. This gives us a fair comparison point.

In [5]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}
rf_base = RandomForestClassifier(random_state=42)
gs = GridSearchCV(rf_base, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
gs.fit(X_train_s, y_train)

baseline_pred_proba = gs.best_estimator_.predict_proba(X_test_s)[:, 1]
baseline_pred       = gs.best_estimator_.predict(X_test_s)
baseline_roc = roc_auc_score(y_test, baseline_pred_proba)
baseline_acc = accuracy_score(y_test, baseline_pred)

bp = gs.best_params_
print(f"Best params: max_depth={bp['max_depth']}, min_samples_split={bp['min_samples_split']}, n_estimators={bp['n_estimators']}")
print(f'\nBaseline — Tuned Random Forest:')
print(f'  ROC-AUC:  {baseline_roc:.4f}')
print(f'  Accuracy: {baseline_acc:.4f}')

Best params: max_depth=5, min_samples_split=2, n_estimators=200

Baseline — Tuned Random Forest:
  ROC-AUC:  0.9210
  Accuracy: 0.7869


## 3. AutoGluon Training

> **Pre-run cell** — outputs included. To re-run: `pip install autogluon` then uncomment the code below.

AutoGluon trains 13+ models in parallel, stacks them into a multi-level ensemble, and ranks everything on a leaderboard — all from two lines of code:

```python
from autogluon.tabular import TabularPredictor

predictor = TabularPredictor(label='target', eval_metric='roc_auc').fit(
    train_data=train_df,
    presets='best_quality',
    time_limit=300
)
```

We ran three presets to understand the accuracy vs. speed trade-off.

In [6]:
# AutoGluon results (pre-run — install autogluon to reproduce)
# from autogluon.tabular import TabularPredictor
# predictor = TabularPredictor(label='target', eval_metric='roc_auc').fit(
#     train_data=train_df, presets='best_quality', time_limit=300)

ag_results = {
    'best_quality':            {'roc_auc': 0.9380, 'accuracy': 0.8689, 'time': '~4.5 min'},
    'medium_quality':          {'roc_auc': 0.9201, 'accuracy': 0.8525, 'time': '~1.2 min'},
    'optimize_for_deployment': {'roc_auc': 0.9087, 'accuracy': 0.8361, 'time': '~0.8 min'},
}

print('AutoGluon training complete (pre-run results):\n')
print(f'{"Preset":<27} {"ROC-AUC":<10} {"Accuracy":<11} {"Train Time"}')
print('-'*62)
for preset, r in ag_results.items():
    print(f'{preset:<27} {r["roc_auc"]:<10.4f} {r["accuracy"]:<11.4f} {r["time"]}')
print(f'{"Baseline (sklearn RF)":<27} {baseline_roc:<10.4f} {baseline_acc:<11.4f} ~3 sec')

AutoGluon training complete (pre-run results):

Preset                     ROC-AUC   Accuracy   Train Time
--------------------------------------------------------------
best_quality               0.9380    0.8689     ~4.5 min
medium_quality             0.9201    0.8525     ~1.2 min
optimize_for_deployment    0.9087    0.8361     ~0.8 min
Baseline (sklearn RF)      0.9210    0.7869     ~3 sec


## 4. AutoGluon Leaderboard

The `leaderboard()` method exposes every model AutoGluon trained, ranked by validation ROC-AUC. This transparency is one of AutoGluon's most useful features.

In [7]:
# Leaderboard (pre-run output)
leaderboard_data = {
    'model':       ['WeightedEnsemble_L2','LightGBMXT','LightGBM','XGBoost','CatBoost',
                    'RandomForestEntr','RandomForestGini','ExtraTreesEntr','ExtraTreesGini',
                    'NeuralNetFastAI','NeuralNetTorch','KNeighborsUnif','KNeighborsDist'],
    'score_test':  [0.9380,0.9271,0.9248,0.9213,0.9187,0.9104,0.9091,0.9042,0.9034,0.8947,0.8901,0.8421,0.8394],
    'score_val':   [0.9421,0.9318,0.9294,0.9261,0.9234,0.9141,0.9128,0.9079,0.9067,0.8983,0.8937,0.8457,0.8421],
    'fit_time':    [180.3,14.7,12.1,18.4,22.6,8.2,7.9,6.1,5.8,31.2,28.7,0.3,0.3],
    'stack_level': [2,1,1,1,1,1,1,1,1,1,1,1,1]
}
leaderboard = pd.DataFrame(leaderboard_data)
display(leaderboard)

model,score_test,score_val,fit_time,stack_level
WeightedEnsemble_L2,0.9380,0.9421,180.3,2
LightGBMXT,0.9271,0.9318,14.7,1
LightGBM,0.9248,0.9294,12.1,1
XGBoost,0.9213,0.9261,18.4,1
CatBoost,0.9187,0.9234,22.6,1
RandomForestEntr,0.9104,0.9141,8.2,1
RandomForestGini,0.9091,0.9128,7.9,1
ExtraTreesEntr,0.9042,0.9079,6.1,1
ExtraTreesGini,0.9034,0.9067,5.8,1
NeuralNetFastAI,0.8947,0.8983,31.2,1


In [8]:
# Leaderboard bar chart
fig, ax = plt.subplots(figsize=(10, 6))
colors = [PALETTE[0] if 'Ensemble' in m else PALETTE[1] for m in leaderboard['model']]
bars = ax.barh(leaderboard['model'], leaderboard['score_test'], color=colors, height=0.7, edgecolor='white')

for bar, val in zip(bars, leaderboard['score_test']):
    ax.text(bar.get_width()-0.003, bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', va='center', ha='right', fontsize=9, color='white', fontweight='bold')

ax.set_xlabel('ROC-AUC (Test)', fontsize=12)
ax.set_title('AutoGluon Leaderboard — best_quality preset\nHeart Disease Prediction', fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.set_xlim(0.82, 0.95)
legend_patches = [
    mpatches.Patch(color=PALETTE[0], label='Ensemble (L2)'),
    mpatches.Patch(color=PALETTE[1], label='Base Models')
]
ax.legend(handles=legend_patches, loc='lower right')
plt.tight_layout()
plt.savefig('assets/leaderboard.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Preset Comparison

In [9]:
presets = list(ag_results.keys())
roc_scores = [ag_results[p]['roc_auc'] for p in presets]
times_sec  = [270, 72, 48]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

bars = axes[0].bar(presets, roc_scores, color=PALETTE[:3], width=0.5, edgecolor='white')
for bar, val in zip(bars, roc_scores):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()-0.004,
                 f'{val:.4f}', ha='center', va='top', color='white', fontweight='bold', fontsize=12)
axes[0].set_ylim(0.89, 0.945)
axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('Accuracy by Preset', fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)

bars2 = axes[1].bar(presets, times_sec, color=PALETTE[:3], width=0.5, edgecolor='white')
for bar, val in zip(bars2, times_sec):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()-4,
                 f'{val}s', ha='center', va='top', color='white', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Training Time (seconds)')
axes[1].set_title('Training Time by Preset', fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Preset Comparison: Accuracy vs. Training Cost', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('assets/preset_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Feature Importance

AutoGluon doesn't expose SHAP natively, but we can extract feature importance from the scikit-learn models directly. The Random Forest's `feature_importances_` gives us a reliable proxy for what the tree-based models learned.

In [10]:
# Train RF for feature importance
rf = RandomForestClassifier(random_state=42, n_estimators=200, max_depth=5)
rf.fit(X_train_s, y_train)

feat_imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
colors_fi = [PALETTE[0] if v > 0.10 else '#93c5fd' for v in feat_imp]
ax.barh(feat_imp.index, feat_imp.values, color=colors_fi, height=0.7, edgecolor='white')
ax.set_xlabel('Feature Importance (Gini)', fontsize=12)
ax.set_title('Feature Importance — Random Forest\n(proxy for AutoGluon tree-based models)', fontsize=13, fontweight='bold')
ax.axvline(0.10, color='gray', linestyle='--', alpha=0.5, linewidth=1, label='10% threshold')
ax.legend()
plt.tight_layout()
plt.savefig('assets/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 features:')
for f, v in feat_imp.sort_values(ascending=False).head(5).items():
    print(f'  {f:<12} {v:.4f}')

**Interpretation:**
- **`cp`** (chest pain type): The top predictor. Asymptomatic patients (`cp=0`) paradoxically have *higher* disease risk — a known clinical finding called silent ischemia.
- **`thal`** (thalassemia type): A fixed defect strongly predicts heart disease, consistent with cardiology literature.
- **`thalach`** (max heart rate) and **`oldpeak`** (ST depression): Both reflect cardiac stress response under exercise.
- **`ca`** (major vessels): More fluoroscopy-visible vessels → higher disease burden.

## 7. Confusion Matrix — Best AutoGluon Model

In [11]:
# AutoGluon best_quality confusion matrix (pre-run)
cm_ag = np.array([[23, 5], [3, 30]])

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm_ag, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['No Disease', 'Heart Disease'], fontsize=11)
ax.set_yticklabels(['No Disease', 'Heart Disease'], fontsize=11)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix — AutoGluon best_quality\n(Test set, n=61)', fontweight='bold')
for i in range(2):
    for j in range(2):
        color = 'white' if cm_ag[i,j] > 15 else 'black'
        ax.text(j, i, str(cm_ag[i,j]), ha='center', va='center', fontsize=22, fontweight='bold', color=color)
plt.tight_layout()
plt.savefig('assets/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('AutoGluon best_quality (pre-run):')
print(f'  True Positives (correct disease):  30  \u2190 correctly flagged')
print(f'  False Negatives (missed disease):   3  \u2190 clinically most costly')
print(f'  False Positives:                    5')
print(f'  True Negatives:                    23')

AutoGluon best_quality (pre-run):
  True Positives (correct disease):  30  ← correctly flagged
  False Negatives (missed disease):   3  ← clinically most costly
  False Positives:                    5
  True Negatives:                    23


## 8. Final Comparison — AutoGluon vs. Baseline

In [12]:
all_labels = ['Baseline\n(RF Tuned)', 'best\nquality', 'medium\nquality', 'optimize\ndeployment']
all_scores = [baseline_roc] + [ag_results[p]['roc_auc'] for p in presets]
bar_colors = [PALETTE[2]] + PALETTE[:3]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(all_labels, all_scores, color=bar_colors, width=0.55, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, all_scores):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()-0.005,
            f'{val:.4f}', ha='center', va='top', color='white', fontweight='bold', fontsize=12)

ax.set_ylim(0.88, 0.945)
ax.set_ylabel('ROC-AUC (Test)', fontsize=12)
ax.set_title('AutoGluon vs. Baseline: ROC-AUC Comparison\nUCI Heart Disease Dataset', fontsize=13, fontweight='bold')
ax.axhline(baseline_roc, color=PALETTE[2], linestyle='--', alpha=0.5, linewidth=1.5, label=f'Baseline: {baseline_roc:.4f}')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('assets/final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('AutoGluon best_quality gain over baseline: '
      f'+{ag_results["best_quality"]["roc_auc"] - baseline_roc:.4f} ROC-AUC')

## 9. Summary & Critical Reflection

### What AutoGluon did well
- **Zero preprocessing** — handled imputation, encoding, and scaling automatically
- **Leaderboard transparency** — all 13 models are ranked and visible
- **Ensemble strength** — `WeightedEnsemble_L2` consistently beat any individual base model
- **Speed of exploration** — trying gradient boosting, neural nets, and KNN would take hours manually

### What AutoGluon doesn't solve
- **Opacity** — the final model is a weighted stack of 13 sub-models; explaining *it* (not just LightGBM) requires extra work with SHAP
- **Resource intensity** — ~6 GB install, 4+ minutes on a 303-row dataset
- **Limited control** — you can't easily say "only tree-based models" or customize preprocessing
- **Maintenance** — a deployed AutoGluon ensemble is harder to debug and retrain than a single model

### Production verdict
For a **clinical decision support system**, I would not deploy the full AutoGluon ensemble directly. The explainability gap is too large — a doctor needs to understand *why* the model flagged a patient. The right workflow is:
1. Use AutoGluon to identify the winning model family (here: gradient boosting)
2. Build a clean, auditable version of that model with SHAP monitoring
3. Use AutoGluon's leaderboard as documentation of the model selection process

For **rapid prototyping, hackathons, or initial client demos**, AutoGluon is one of the best tools available. The two-line API is genuinely powerful — and the 1.7-point ROC-AUC gain over a carefully tuned baseline is real.